In [1]:
%pip install -U torch transformers datasets accelerate evaluate scikit-learn huggingface_hub torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 41.0 MB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 26.9 MB/s  0:00:15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 51.1 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 15.1 MB/s  0:00:19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 35.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 46.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 31.2 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 MB 51.5 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.0/151.0 MB 58.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 MB 19.5 MB/s  0:00:13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 43.2 MB/s  0:00:00


In [ ]:
try:
    from huggingface_hub import HfApi
    user_info = HfApi().whoami()
    print(f"✅ Logged in successfully as: {user_info['name']}")
except Exception as e:
    print("❌ Not logged in or token is invalid.")

❌ Not logged in or token is invalid.


In [2]:
from huggingface_hub import login

login()

In [3]:
import os
import numpy as np
import pandas as pd
import torch

#from datasets import Dataset, Datasetdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
CSV_path = "bengali_sentences_final.csv"

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

print("CUDA available:", torch.cuda.is_available())
print("PyTorch device:", "cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
PyTorch device: cuda


In [4]:
df = pd.read_csv("bengali_sentences_final.csv")

# Select columns: sentence (1st col in selection), positive, negative, question, happiness(7th col), sadness(10th col)
selected_df = pd.concat([
    df.iloc[:, 0],   # sentence
    df.iloc[:, 1],   # positive
    df.iloc[:, 2],   # negative
    df.iloc[:, 3],   # question
    df.iloc[:, 4],   # happiness (7th col, index 6)
    df.iloc[:, 5]    # sadness (10th col, index 9)
], axis=1).fillna(0)

# Rename columns
selected_df.columns = ['sentence', 'Positive', 'Negative', 'Question', 'Happiness', 'Sadness']

print(selected_df.head())

                                            sentence  Positive  Negative  \
0                          পড়ছি আর নিজের মনে হাসছি।         1         0   
1        বাবা টেনি, তুমি আমায় ধনে-প্রাণে বাঁচিয়েছ।         1         0   
2  কাঙ্গা নেপালীর ছেলে, জান দিয়ে দেবে, কিন্তু মন...         0         1   
3                  এ যাত্রা আপনি ওকে নিয়ে যাবেন না।         0         1   
4                                       কিছু কি নেই?         0         1   

   Question  Happiness  Sadness  
0         0          1        0  
1         0          1        0  
2         0          0        0  
3         0          0        0  
4         1          0        0  


In [5]:
hf_dataset = Dataset.from_pandas(selected_df)

def process_labels(example):
    example["labels"] = [
        float(example["Positive"]),
        float(example["Happiness"]),
        float(example["Negative"]),
        float(example["Sadness"]),
        float(example["Question"])
    ]
    return example

hf_dataset = hf_dataset.map(process_labels)


# Remove original separate columns
hf_dataset = hf_dataset.remove_columns(['Positive', 'Happiness', 'Negative',  'Sadness', 'Question'])
hf_dataset
#print(hf_dataset[0])

Map:   0%|          | 0/5182 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence', 'labels'],
    num_rows: 5182
})

In [6]:
# Convert the Hugging Face dataset to a DataFrame
df1 = hf_dataset.to_pandas()

# Remove rows with no active labels
#df1 = df[df["labels"].apply(lambda x: sum(x) > 0)].reset_index(drop=True)


train_df, temp_df = train_test_split(
    df1,
    test_size = 0.3,
    random_state = 42,

)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,

)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

Training samples: 3627
Validation samples: 777
Test samples: 778


In [7]:
print("Train columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

print("\nFirst training example:")
print(train_df.iloc[0])

print("\nData types:")
print(train_df.dtypes)

Train columns: ['sentence', 'labels']
Validation columns: ['sentence', 'labels']
Test columns: ['sentence', 'labels']

First training example:
sentence    কিন্তু মরা মানুষকেও যে জাগিয়ে তুলতে পারে সে হ...
labels                              [1.0, 0.0, 0.0, 0.0, 0.0]
Name: 3722, dtype: object

Data types:
sentence    object
labels      object
dtype: object


In [8]:
from datasets import Dataset, DatasetDict, Features
from datasets import Value, Sequence

# Define the dataset schema
features = Features({
    "sentence": Value("string"),
    "labels": Sequence(Value("float32"), length=5)
})

# Convert each split
hf_dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["sentence", "labels"]],
        features=features,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        val_df[["sentence", "labels"]],
        features=features,
        preserve_index=False
    ),
    "test": Dataset.from_pandas(
        test_df[["sentence", "labels"]],
        features=features,
        preserve_index=False
    )
})

print(hf_dataset)
print("\nOne training example:")
print(hf_dataset["train"][0])

print("\nDataset features:")
print(hf_dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'labels'],
        num_rows: 3627
    })
    validation: Dataset({
        features: ['sentence', 'labels'],
        num_rows: 777
    })
    test: Dataset({
        features: ['sentence', 'labels'],
        num_rows: 778
    })
})

One training example:
{'sentence': 'কিন্তু মরা মানুষকেও যে জাগিয়ে তুলতে পারে সে হল বাঘের ডাক।', 'labels': [1.0, 0.0, 0.0, 0.0, 0.0]}

Dataset features:
{'sentence': Value('string'), 'labels': List(Value('float32'), length=5)}


In [18]:
# Pretrained XLM-RoBERTa model
BASE_MODEL = "FacebookAI/xlm-roberta-base"

# New Hugging Face repository
NEW_MODEL_REPO = "Subha95/bengali-sentiment-xlmr-v2"



In [19]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Tokenizer loaded:", BASE_MODEL)
print(tokenizer("বাংলা ভাষায় একটি উদাহরণ"))

Tokenizer loaded: FacebookAI/xlm-roberta-base
{'input_ids': [0, 38732, 73765, 2801, 14536, 16447, 30010, 10062, 70145, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [20]:
label_names = [
    "Positive",
    "Happiness",
    "Negative",
    "Sadness",
    "Question"
]
id2label = {
    i: name for i, name in enumerate(label_names)
}

label2id = {
    name: i for i, name in enumerate(label_names)
}

In [21]:
MAX_LENGTH = 128
TEXT_COLUMN = "sentence"

def tokenize_function(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation = True,
        max_length = MAX_LENGTH
    )

tokenized_dataset = hf_dataset.map(
    tokenize_function,
    batched = True,
     remove_columns=[TEXT_COLUMN],
)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/777 [00:00<?, ? examples/s]

Map:   0%|          | 0/778 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 3627
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 777
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 778
    })
})
{'labels': [1.0, 0.0, 0.0, 0.0, 0.0], 'input_ids': [0, 46729, 13705, 5534, 58041, 2937, 4876, 14510, 30414, 8751, 21264, 6, 103771, 3495, 40449, 22540, 86507, 7802, 38477, 896, 46281, 2784, 125, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels = len(label_names),
    problem_type = "multi_label_classification",
    id2label = id2label,
    label2id=label2id
)
print("Model loaded:", BASE_MODEL)
print("Number of classes:", model.config.num_labels)
print("Model label mapping:", model.config.id2label)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: FacebookAI/xlm-roberta-base
Number of classes: 5
Model label mapping: {0: 'Positive', 1: 'Happiness', 2: 'Negative', 3: 'Sadness', 4: 'Question'}


In [23]:

import numpy as np
from scipy.special import expit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score
)

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred

    # Convert logits to probabilities using sigmoid
    probabilities = expit(logits)

    # Convert probabilities into binary predictions
    predictions = (probabilities >= 0.5).astype(int)

    # Ensure labels are in the expected binary format
    true_labels = np.asarray(true_labels).astype(int)

    # Macro metrics: calculate each label's score,
    # then give all five labels equal weight
    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    # Micro F1: calculate overall performance across all labels
    f1_micro = f1_score(
        true_labels,
        predictions,
        average="micro",
        zero_division=0
    )

    # Exact match: all five labels must be correct
    exact_match_accuracy = accuracy_score(
        true_labels,
        predictions
    )

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "exact_match_accuracy": exact_match_accuracy
    }

In [24]:
import transformers
import inspect

print("Transformers version:", transformers.__version__)
print(inspect.signature(TrainingArguments.__init__))

Transformers version: 5.22.2
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_co

In [25]:
import math

# Local output directory
OUTPUT_DIR = "./bengali-sentiment-xlmr-output"

# Training configuration
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
EPOCHS = 5

# Calculate total training steps
steps_per_epoch = math.ceil(
    math.ceil(len(tokenized_dataset["train"]) / BATCH_SIZE)
    / GRADIENT_ACCUMULATION
)

total_training_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * 0.1)

print("Steps per epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    # Evaluation and checkpoints
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    save_total_limit=2,

    # Hugging Face Hub upload
    push_to_hub=True,
    hub_model_id=NEW_MODEL_REPO,

    # Reproducibility and logging
    seed=42,
    data_seed=42,
    logging_steps=20,
    report_to="none",

    # Precision
    fp16=False,
)

Steps per epoch: 227
Total training steps: 1135
Warmup steps: 113


In [26]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [27]:
train_result = trainer.train()

print("\nTraining completed.")
print(train_result.metrics)

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Precision Macro,Recall Macro,Exact Match Accuracy
1,0.497872,0.229404,0.757600,0.824903,0.874872,0.702339,0.617761
2,0.357463,0.171115,0.868766,0.887393,0.900611,0.842332,0.760618
3,0.300177,0.150913,0.889337,0.906872,0.932321,0.858341,0.792793
4,0.197753,0.146174,0.890568,0.908108,0.930281,0.863885,0.792793
5,0.214121,0.137017,0.902030,0.914439,0.929238,0.879130,0.803089


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed.
{'train_runtime': 582.3473, 'train_samples_per_second': 31.141, 'train_steps_per_second': 1.949, 'total_flos': 274420916732268.0, 'train_loss': 0.4120253926331776, 'epoch': 5.0}


In [34]:
import os
import glob
from safetensors.torch import load_file

best_checkpoint = trainer.state.best_model_checkpoint

print("Best checkpoint:", best_checkpoint)

checkpoint_files = glob.glob(
    os.path.join(best_checkpoint, "*.safetensors")
)

for file in checkpoint_files:
    state_dict = load_file(file, device="cpu")

    layernorm_keys = [
        key for key in state_dict.keys()
        if "embeddings.LayerNorm" in key
    ]

    print("\nFile:", file)
    print("LayerNorm keys:", layernorm_keys)

Best checkpoint: ./bengali-sentiment-xlmr-output/checkpoint-1135

File: ./bengali-sentiment-xlmr-output/checkpoint-1135/model.safetensors
LayerNorm keys: ['roberta.embeddings.LayerNorm.bias', 'roberta.embeddings.LayerNorm.weight']


In [35]:
validation_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["validation"]
)

print("Validation results:")
for metric, value in validation_results.items():
    print(f"{metric}: {value}")

Training Loss,Validation Loss,Epoch,F1 Macro,F1 Micro,Precision Macro,Recall Macro,Exact Match Accuracy
0.214121,0.137017,5,0.902030,0.914439,0.929238,0.879130,0.803089


Validation results:
eval_loss: 0.13701699674129486
eval_f1_macro: 0.9020296167386703
eval_f1_micro: 0.9144385026737968
eval_precision_macro: 0.9292381939621519
eval_recall_macro: 0.879129701188878
eval_exact_match_accuracy: 0.803088803088803


In [36]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

print("\nFinal test results:")
for metric, value in test_results.items():
    print(f"{metric}: {value}")

Training Loss,Validation Loss,Epoch,F1 Macro,F1 Micro,Precision Macro,Recall Macro,Exact Match Accuracy
0.214121,0.115875,5,0.915594,0.931738,0.944295,0.892422,0.834190



Final test results:
test_loss: 0.11587458848953247
test_f1_macro: 0.9155939908563067
test_f1_micro: 0.9317375886524822
test_precision_macro: 0.9442954486191718
test_recall_macro: 0.8924215547602644
test_exact_match_accuracy: 0.8341902313624678


In [37]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model and tokenizer saved to:", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Model and tokenizer saved to: ./bengali-sentiment-xlmr-output


In [38]:
trainer.push_to_hub(
    commit_message="Upload fine-tuned Bengali sentiment model"
)

tokenizer.push_to_hub(NEW_MODEL_REPO)

print("Upload requested for:", NEW_MODEL_REPO)
print(f"https://huggingface.co/{NEW_MODEL_REPO}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9wd3cyzm/tokenizer.json:  93%|#########3| 15.9MB / 17.1MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


Upload requested for: Subha95/bengali-sentiment-xlmr-v2
https://huggingface.co/Subha95/bengali-sentiment-xlmr-v2


In [33]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=NEW_MODEL_REPO,
    tokenizer=NEW_MODEL_REPO,
    device=0 if torch.cuda.is_available() else -1,
)

sample_texts = [
    "আজ আমার খুব ভালো লাগছে",
    "আমি খুব দুঃখিত",
    "এটি মোটেও ভালো নয়",
    "তুমি কি আজ আসবে",
]

predictions = classifier(
    sample_texts,
    truncation=True,
    max_length=MAX_LENGTH,
)

for text, prediction in zip(sample_texts, predictions):
    print(f"Text: {text}")
    print(f"Predicted label: {prediction['label']}")
    print(f"Confidence: {prediction['score']:.4f}")
    print("-" * 50)

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

Text: আজ আমার খুব ভালো লাগছে
Predicted label: Happiness
Confidence: 0.9874
--------------------------------------------------
Text: আমি খুব দুঃখিত
Predicted label: Sadness
Confidence: 0.9740
--------------------------------------------------
Text: এটি মোটেও ভালো নয়
Predicted label: Negative
Confidence: 0.9904
--------------------------------------------------
Text: তুমি কি আজ আসবে
Predicted label: Question
Confidence: 0.9733
--------------------------------------------------
